# Vecka 5 – Kapitel 5: kod

Koduppgifterna 8–9 (7 = avskrift av kapitlets exempel).

## Uppgift 8 – förklara PCA-koden

Koden skapar slumpdata med 3 features, komprimerar till 2 med PCA och försöker återskapa 3D igen. `np.allclose` blir `False` eftersom PCA kastar bort en dimension – informationen går inte att få tillbaka exakt.

In [1]:
import numpy as np
from sklearn.decomposition import PCA

X = np.random.rand(1000, 3)                 # 3 features
pca = PCA(n_components=2)
X2D = pca.fit_transform(X)                   # -> 2 dimensioner
X3D_inv = pca.inverse_transform(X2D)         # tillbaka till 3, men info förlorad
print("Bevarad varians:", pca.explained_variance_ratio_.sum().round(3))
print("Exakt återskapad:", np.allclose(X3D_inv, X))

Bevarad varians: 0.699
Exakt återskapad: False


## Uppgift 9 – PCA på car_price_dataset.csv innan ML

Jämför RandomForest med och utan PCA. Allt i en pipeline så scaler/PCA fittas bara på träningsdatan (inget läckage).

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

df = pd.read_csv("../dataset/car_price_dataset.csv", sep=";")
X = pd.get_dummies(df.drop(columns=["Price"]))
y = df["Price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
print(f"Utan PCA:  R2 {r2_score(y_test, rf.predict(X_test)):.4f}   ({X_train.shape[1]} features)")

for k in [2, 5, 10]:
    pipe = make_pipeline(StandardScaler(), PCA(n_components=k),
                         RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
    pipe.fit(X_train, y_train)
    print(f"PCA k={k:2d}:   R2 {r2_score(y_test, pipe.predict(X_test)):.4f}")

Utan PCA:  R2 0.9878   (52 features)
PCA k= 2:   R2 -0.0245
PCA k= 5:   R2 0.5143
PCA k=10:   R2 0.7556


**Slutsats.** PCA *försämrar* resultatet här: RandomForest hanterar 52 features utan problem, medan PCA slänger bort signal på vägen (R² faller från 0.99 till 0.76 vid k=10). PCA lönar sig när features är många och starkt korrelerade eller när man behöver snabbare träning – inte som standardsteg.